# 🚗 Car Market Trends Analysis — Notebook 3
## Visualizations — All Charts Saved as PNG

**Project:** Car Market Trends Analysis with Car Dekho Data  
**Goal:** Generate, display, and save all 10 publication-quality charts to `report_images/`.

---
### Charts in this notebook
| # | Chart | File |
|---|-------|------|
| 1 | Selling Price Distribution | `01_price_distribution.png` |
| 2 | Price by Fuel Type (box) | `02_price_by_fuel.png` |
| 3 | Price by Seller Type (box) | `03_price_by_seller.png` |
| 4 | Price by Transmission (box) | `04_price_by_transmission.png` |
| 5 | Car Age vs Selling Price | `05_age_vs_price.png` |
| 6 | Km Driven vs Selling Price | `06_kms_vs_price.png` |
| 7 | Fuel Type Share (pie) | `07_fuel_share.png` |
| 8 | Avg Price by Year (line) | `08_avg_price_by_year.png` |
| 9 | Top 10 Models (bar) | `09_top_models.png` |
| 10 | Correlation Heatmap | `10_correlation_heatmap.png` |

---
## Setup

In [ ]:
import os, sys, warnings
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — saves files without opening windows
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# Colour constants
BLUE   = '#2563eb'
PURPLE = '#7c5cd8'
ORANGE = '#ea580c'

# Output folder for saved charts
OUT_DIR = os.path.join('..', 'report_images')
os.makedirs(OUT_DIR, exist_ok=True)

# Load data
CLEANED_PATH = os.path.join('..', 'data', 'car_dekho_cleaned.csv')
if os.path.exists(CLEANED_PATH):
    df = pd.read_csv(CLEANED_PATH)
else:
    sys.path.insert(0, os.path.join('..', 'src'))
    from data_cleaning import get_clean_data
    df = get_clean_data()

print(f'Dataset loaded: {df.shape}')
print(f'Charts will be saved to: {os.path.abspath(OUT_DIR)}')

def save_and_show(fig, name):
    """Save figure to report_images/ and display it inline."""
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved: {path}')
    # Re-render for notebook display
    matplotlib.use('module://matplotlib_inline.backend_inline')
    fig.canvas.draw()
    plt.show()
    matplotlib.use('Agg')

---
## Chart 1 — Selling Price Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df['Selling_Price'], bins=35, kde=True, color=BLUE, ax=ax)
ax.set_title('Selling Price Distribution', fontsize=13)
ax.set_xlabel('Selling Price (Rs Lakhs)')
ax.set_ylabel('Number of Listings')
plt.tight_layout()
save_and_show(fig, '01_price_distribution.png')

---
## Chart 2 — Selling Price by Fuel Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order = df.groupby('Fuel_Type')['Selling_Price'].median().sort_values(ascending=False).index

sns.barplot(data=df, x='Fuel_Type', y='Selling_Price', order=order,
            estimator='mean', palette='Set2', errorbar='sd', ax=axes[0])
axes[0].set_title('Mean Selling Price by Fuel Type')
axes[0].set_ylabel('Avg Price (Rs Lakhs)')

sns.boxplot(data=df, x='Fuel_Type', y='Selling_Price', order=order,
            palette='Set2', ax=axes[1])
axes[1].set_title('Price Distribution by Fuel Type')
axes[1].set_ylabel('Selling Price (Rs Lakhs)')

plt.suptitle('Chart 2 — Selling Price by Fuel Type', fontsize=13)
plt.tight_layout()
save_and_show(fig, '02_price_by_fuel.png')

---
## Chart 3 — Selling Price by Seller Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
sns.barplot(data=df, x='Seller_Type', y='Selling_Price',
            estimator='mean', palette='pastel', errorbar='sd', ax=axes[0])
axes[0].set_title('Mean Selling Price by Seller Type')
axes[0].set_ylabel('Avg Price (Rs Lakhs)')

sns.boxplot(data=df, x='Seller_Type', y='Selling_Price',
            palette='pastel', ax=axes[1])
axes[1].set_title('Price Distribution by Seller Type')
axes[1].set_ylabel('Selling Price (Rs Lakhs)')

plt.tight_layout()
save_and_show(fig, '03_price_by_seller.png')

---
## Chart 4 — Selling Price by Transmission

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
sns.barplot(data=df, x='Transmission', y='Selling_Price',
            estimator='mean', palette=[BLUE, PURPLE],
            errorbar='sd', ax=axes[0])
axes[0].set_title('Mean Selling Price by Transmission')
axes[0].set_ylabel('Avg Price (Rs Lakhs)')

sns.violinplot(data=df, x='Transmission', y='Selling_Price',
               palette=[BLUE, PURPLE], inner='quartile', ax=axes[1])
axes[1].set_title('Price Distribution by Transmission')
axes[1].set_ylabel('Selling Price (Rs Lakhs)')

plt.tight_layout()
save_and_show(fig, '04_price_by_transmission.png')

---
## Chart 5 — Car Age vs Selling Price

In [ ]:
corr = df['Car_Age'].corr(df['Selling_Price'])
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x='Car_Age', y='Selling_Price',
                hue='Fuel_Type', alpha=0.65, s=60, palette='Set1', ax=ax)
z = np.polyfit(df['Car_Age'], df['Selling_Price'], 1)
x_line = np.linspace(df['Car_Age'].min(), df['Car_Age'].max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), 'k--', linewidth=1.5, label='Trend')
ax.legend(title='Fuel Type', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_title(f'Car Age vs Selling Price  (r = {corr:.2f})', fontsize=13)
ax.set_xlabel('Car Age (Years)')
ax.set_ylabel('Selling Price (Rs Lakhs)')
plt.tight_layout()
save_and_show(fig, '05_age_vs_price.png')

---
## Chart 6 — Kilometres Driven vs Selling Price

In [ ]:
corr = df['Kms_Driven'].corr(df['Selling_Price'])
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x='Kms_Driven', y='Selling_Price',
                hue='Fuel_Type', alpha=0.55, s=55, palette='Set2', ax=ax)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title(f'Km Driven vs Selling Price  (r = {corr:.2f})', fontsize=13)
ax.set_xlabel('Kilometres Driven')
ax.set_ylabel('Selling Price (Rs Lakhs)')
ax.legend(title='Fuel Type', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
save_and_show(fig, '06_kms_vs_price.png')

---
## Chart 7 — Fuel Type Share (Pie)

In [ ]:
counts = df['Fuel_Type'].value_counts()
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
       startangle=140, colors=sns.color_palette('Set2', len(counts)))
ax.set_title('Fuel Type Share in Used-Vehicle Market', fontsize=13)
plt.tight_layout()
save_and_show(fig, '07_fuel_share.png')

---
## Chart 8 — Average Selling Price by Year

In [ ]:
yearly = df.groupby('Year')['Selling_Price'].mean().reset_index()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly['Year'], yearly['Selling_Price'],
        marker='o', color=BLUE, linewidth=2.2, markersize=7)
ax.fill_between(yearly['Year'], yearly['Selling_Price'], alpha=0.1, color=BLUE)
for _, row in yearly.iterrows():
    ax.annotate(f"{row['Selling_Price']:.1f}",
                xy=(row['Year'], row['Selling_Price']),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8)
ax.set_title('Average Selling Price by Manufacture Year', fontsize=13)
ax.set_xlabel('Year of Manufacture')
ax.set_ylabel('Avg Selling Price (Rs Lakhs)')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.xticks(rotation=45)
plt.tight_layout()
save_and_show(fig, '08_avg_price_by_year.png')

---
## Chart 9 — Top 10 Models by Listing Count

In [ ]:
top10 = df['Car_Name'].value_counts().head(10).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
top10.plot(kind='barh', color=PURPLE, ax=ax)
for i, v in enumerate(top10.values):
    ax.text(v + 0.1, i, str(v), va='center', fontsize=9)
ax.set_title('Top 10 Most Listed Car / Bike Models', fontsize=13)
ax.set_xlabel('Number of Listings')
plt.tight_layout()
save_and_show(fig, '09_top_models.png')

---
## Chart 10 — Correlation Heatmap

In [ ]:
num_cols = ['Selling_Price', 'Present_Price', 'Kms_Driven',
            'Car_Age', 'Year', 'Owner', 'Price_Depreciation']
corr = df[num_cols].corr().round(2)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=13, pad=14)
plt.tight_layout()
save_and_show(fig, '10_correlation_heatmap.png')

---
## All Charts Saved ✅

All 10 charts are now saved in `report_images/`. You can insert them into your project report or presentation.

> **Next step:** Open `04_machine_learning.ipynb` to train and evaluate the prediction model.